# Scenario: "red car behind the bus"

Parks a bus, a red car 12 m behind it, and a second identical red car off to the
side as a distractor. Records 15 s of RGB + ground-truth scene graphs.

Run cells top to bottom. Check the preview in cell 5 before recording.

In [1]:
import carla
import math
import os
import json
import queue
import numpy as np
from PIL import Image

In [2]:
client = carla.Client('localhost', 2000)
client.set_timeout(200.0)
world  = client.reload_world()
bp_lib = world.get_blueprint_library()
spawn_points = world.get_map().get_spawn_points()

## 1. Synchronous mode — 20 fps

In [8]:
FPS = 20

settings = world.get_settings() 
settings.synchronous_mode = True
settings.fixed_delta_seconds = 1.0 / FPS
world.apply_settings(settings)

12672

## 2. Spawn the three vehicles

In [9]:
RED = '180,20,20'

def find_bp(*names):
    for n in names:
        hits = bp_lib.filter(n)
        if hits:
            return hits[0]
    raise RuntimeError(f'no blueprint matching {names}')

# clear anything left over from a previous run
for a in world.get_actors().filter('*vehicle*'):
    a.destroy()

anchor = spawn_points[0]
fwd    = anchor.get_forward_vector()
right  = anchor.get_right_vector()

# the bus
bus_bp = find_bp('vehicle.mitsubishi.fusorosa', 'vehicle.volkswagen.t2', '*bus*')
bus = world.spawn_actor(bus_bp, anchor)

# red car 12 m BEHIND the bus
behind_tf = carla.Transform(
    carla.Location(x=anchor.location.x - fwd.x*12.0,
                   y=anchor.location.y - fwd.y*12.0,
                   z=anchor.location.z + 0.3),
    anchor.rotation)
tbp = find_bp('vehicle.tesla.model3', 'vehicle.audi.a2')
tbp.set_attribute('color', RED)
target = world.spawn_actor(tbp, behind_tf)

# identical red car off to the side -> the distractor
beside_tf = carla.Transform(
    carla.Location(x=anchor.location.x + fwd.x*6.0 + right.x*5.5,
                   y=anchor.location.y + fwd.y*6.0 + right.y*5.5,
                   z=anchor.location.z + 0.3),
    anchor.rotation)
dbp = find_bp('vehicle.tesla.model3', 'vehicle.audi.a2')
dbp.set_attribute('color', RED)
distractor = world.spawn_actor(dbp, beside_tf)

IDS = {'bus': bus.id, 'target': target.id, 'distractor': distractor.id}
print(IDS)

{'bus': 51, 'target': 52, 'distractor': 53}


## 3. Camera (drone view, elevated and off to one side)

In [4]:
spectator = world.get_spectator()
transform = spectator.get_transform()
location = transform.location
rotation = transform.rotation

# Print coordinates
print(f"X: {location.x}, Y: {location.y}, Z: {location.z}")
print(f"Pitch: {rotation.pitch}, Yaw: {rotation.yaw}, Roll: {rotation.roll}")

X: -172.1965789794922, Y: 183.85911560058594, Z: 27.638051986694336
Pitch: 0.0, Yaw: -45.59214782714844, Roll: 0.0


In [ ]:
WIDTH, HEIGHT, FOV = 1920, 1080, 90

cam_tf = carla.Transform(
    carla.Location(x=-98.53,
                   y=21.73,
                   z=17.28),
    carla.Rotation(pitch=-24.0, yaw=1.48, roll= 0.0))

cam_bp = bp_lib.find('sensor.camera.rgb')
cam_bp.set_attribute('image_size_x', str(WIDTH))
cam_bp.set_attribute('image_size_y', str(HEIGHT))
cam_bp.set_attribute('fov', str(FOV))
# NOTE: no sensor_tick -> one image per world.tick()

camera = world.spawn_actor(cam_bp, cam_tf)

image_queue = queue.Queue()
camera.listen(image_queue.put)

# put the spectator at the same spot so you can eyeball framing in the CARLA window
world.get_spectator().set_transform(cam_tf)

## 4. Projection helpers (from your notebook)

In [11]:
def build_projection_matrix(w, h, fov):
    focal = w / (2.0 * np.tan(fov * np.pi / 360.0))
    K = np.identity(3)
    K[0, 0] = K[1, 1] = focal
    K[0, 2] = w / 2.0
    K[1, 2] = h / 2.0
    return K

def get_image_point(loc, K, w2c):
    point = np.array([loc.x, loc.y, loc.z, 1])
    point_camera = np.dot(w2c, point)
    # UE4 -> standard camera coords: (x, y, z) -> (y, -z, x)
    point_camera = [point_camera[1], -point_camera[2], point_camera[0]]
    point_img = np.dot(K, point_camera)
    point_img[0] /= point_img[2]
    point_img[1] /= point_img[2]
    return point_img[0:2]

def camera_xyz(loc, w2c):
    """3D position in camera frame: x=right, y=down, z=forward (metres)."""
    p = np.dot(w2c, np.array([loc.x, loc.y, loc.z, 1]))
    return np.array([p[1], -p[2], p[0]])

K = build_projection_matrix(WIDTH, HEIGHT, FOV)

## 5. Preview one frame before committing to a full recording

In [12]:
world.tick()
img = image_queue.get()
arr = np.reshape(np.copy(img.raw_data), (img.height, img.width, 4))[:, :, :3][:, :, ::-1]
Image.fromarray(arr).save('preview.png')
print('wrote preview.png -- open it and check all three vehicles are visible')

wrote preview.png -- open it and check all three vehicles are visible


If framing is off, tweak `pitch` / `yaw` / the multipliers in cell 3, re-run cell 3
and this cell. Nothing else needs re-running.

## 6. Record

In [13]:
SECONDS = 15
OUT = 'runs/redcar_behind_bus'
os.makedirs(f'{OUT}/rgb', exist_ok=True)
os.makedirs(f'{OUT}/gt_graphs', exist_ok=True)

COLOR_NAMES = {'180,20,20': 'red'}

def vehicle_class(actor):
    wheels = int(actor.attributes.get('number_of_wheels', 4))
    if 'fusorosa' in actor.type_id or 't2' in actor.type_id:
        return 'bus'
    return 'motorcycle' if wheels == 2 else 'car'

def frame_graph(w2c, frame_id):
    nodes = []
    for a in world.get_actors().filter('*vehicle*'):
        tf = a.get_transform()
        cxyz = camera_xyz(tf.location, w2c)
        if cxyz[2] <= 0.1:            # behind the camera
            continue

        verts = a.bounding_box.get_world_vertices(tf)
        pts = np.array([get_image_point(v, K, w2c) for v in verts])
        x1, y1 = pts[:, 0].min(), pts[:, 1].min()
        x2, y2 = pts[:, 0].max(), pts[:, 1].max()

        raw_col = a.attributes.get('color')
        nodes.append({
            'id': int(a.id),
            'class': vehicle_class(a),
            'color': COLOR_NAMES.get(raw_col),
            'color_rgb': raw_col,
            'box2d': [float(x1), float(y1), float(x2), float(y2)],
            'loc': [tf.location.x, tf.location.y, tf.location.z],
            'yaw': tf.rotation.yaw,
            '_cam': cxyz.tolist(),
        })

    # viewer-centric relations
    edges = []
    for a in nodes:
        for b in nodes:
            if a['id'] == b['id']:
                continue
            dz = a['_cam'][2] - b['_cam'][2]    # +ve => a further away
            dx = a['_cam'][0] - b['_cam'][0]    # +ve => a to the right
            if dz >  2.0: edges.append({'subj': a['id'], 'relation': 'behind',      'obj': b['id']})
            if dz < -2.0: edges.append({'subj': a['id'], 'relation': 'in_front_of', 'obj': b['id']})
            if dx >  1.5: edges.append({'subj': a['id'], 'relation': 'right_of',    'obj': b['id']})
            if dx < -1.5: edges.append({'subj': a['id'], 'relation': 'left_of',     'obj': b['id']})

    for n in nodes:
        n.pop('_cam')

    return {'frame': frame_id, 'convention': 'viewer_centric',
            'ids': IDS, 'nodes': nodes, 'edges': edges}


n_frames = int(SECONDS * FPS)
for i in range(n_frames):
    world.tick()
    image = image_queue.get()

    arr = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))
    Image.fromarray(arr[:, :, :3][:, :, ::-1]).save(f'{OUT}/rgb/{i:06d}.png')

    w2c = np.array(camera.get_transform().get_inverse_matrix())
    with open(f'{OUT}/gt_graphs/{i:06d}.json', 'w') as f:
        json.dump(frame_graph(w2c, i), f)

    if i % FPS == 0:
        print(f'{i//FPS}s / {SECONDS}s')

print('done ->', OUT)

0s / 15s
1s / 15s
2s / 15s
3s / 15s
4s / 15s
5s / 15s
6s / 15s
7s / 15s
8s / 15s
9s / 15s
10s / 15s
11s / 15s
12s / 15s
13s / 15s
14s / 15s
done -> runs/redcar_behind_bus


## 7. Sanity check — is the relation actually true?

In [ ]:
g = json.load(open(f'{OUT}/gt_graphs/000000.json'))
tgt, bus_id = g['ids']['target'], g['ids']['bus']

hit = [e for e in g['edges']
       if e['subj'] == tgt and e['obj'] == bus_id and e['relation'] == 'behind']
print('target is behind bus:', bool(hit))

for n in g['nodes']:
    role = [k for k, v in g['ids'].items() if v == n['id']]
    print(role, n['class'], n['color'], [round(v) for v in n['box2d']])

## 8. Cleanup

In [ ]:
camera.stop()
camera.destroy()
for a in world.get_actors().filter('*vehicle*'):
    a.destroy()

s = world.get_settings()
s.synchronous_mode = False
s.fixed_delta_seconds = None
world.apply_settings(s)
print('cleaned up')